# 4_cmap — classificação de trajetórias com o CMAP e avaliação contra o TerraClass

Etapa 8 do projeto redesenhado (pasta `C:\cmap_ftl\terraclass`). Aplica o CMAP às probabilidades de cada classificador (saídas do `3_inferencia`) e compara com a classificação **data a data** (a classe mais provável em cada ano, empilhada), que é a pós-classificação usada como referência por Reis et al. (2020). As duas classificações usam **as mesmas probabilidades**, então a diferença entre elas é o efeito do CMAP.

Usa as funções do repositório `CMAP_FTL`, as mesmas da dissertação:

- `cmap_classifier`: CMAP, com as matrizes do `4a_matrizes_transicao` (só a volta para Floresta é proibida; trajetórias válidas com o mesmo peso);
- `maxver_from_likelihoods`: classe mais provável de cada data a partir dos mesmos rasters (equivale ao CMAP com todas as transições permitidas);
- `impossible_trajectories` e `compare_classifications`: trajetórias inválidas e discordância entre as duas classificações.

**Avaliação.** A referência é o TerraClass de 2008 e 2010 na grade da área de estudo (`0c_referencia_terraclass`). A comparação usa **todos** os pixels da referência dentro da legenda (não uma amostra), então a acurácia global é a acurácia do mapa; também é calculada sem os pixels de borda da referência. Além das métricas por ano, o par 2008–2010 mede o que o CMAP deve melhorar: a coerência das mudanças (acerto das duas datas, mudanças detectadas e mudanças falsas).

**Incerteza.** Como a comparação é feita na área inteira, não há erro amostral no sentido usual; a variação relevante é espacial. Os intervalos de 95% vêm de um *bootstrap* por blocos de `TAMANHO_BLOCO_PX` × `TAMANHO_BLOCO_PX` pixels, sorteados com reposição, o que respeita a autocorrelação espacial dos erros. As diferenças (CMAP − data a data; CNN − outro classificador) usam os mesmos blocos sorteados nas duas classificações (comparação pareada).

**Confiança das trajetórias.** O CMAP escolhe a trajetória mais provável, mas não diz quanto ela é mais provável que as outras. A seção 6 calcula, em cada pixel, a probabilidade a posteriori da trajetória escolhida e a probabilidade da classe do CMAP em cada data, e confere se essas probabilidades acertam na prática, contra o TerraClass.

Pixels sem probabilidade numa data (máscara do `3_inferencia`) não informam a trajetória naquela data (`mask_mode="ignore"`) e ficam sem classe (0) naquela data.

Lê as saídas do 0b (legenda), 0c, 3_inferencia e 4a e o repositório `C:\cmap_ftl\CMAP_FTL` (só leitura); grava só em `resultados\4_cmap\<NOME_INFERENCIA>\`.

Obs.: Documentação do Código feita com auxílio de IA (Claude Opus 5.5)

## Parâmetros

In [ ]:
# ============================== PARÂMETROS ==============================
PASTA_PROJETO    = r"C:\cmap_ftl\terraclass"
NOME_CONJUNTO    = "6bandas"
PASTA_CONJUNTO   = rf"{PASTA_PROJETO}\resultados\0b_conjunto_treino\{NOME_CONJUNTO}"   # legenda.json
NOME_INFERENCIA  = "6bandas_patch5"                       # subpasta do 3_inferencia
PASTA_INFERENCIA = rf"{PASTA_PROJETO}\resultados\3_inferencia\{NOME_INFERENCIA}"
PASTA_MATRIZES   = rf"{PASTA_PROJETO}\resultados\4a_matrizes_transicao"
PASTA_REFERENCIA = rf"{PASTA_PROJETO}\resultados\0c_referencia_terraclass"
PASTA_REPO       = r"C:\cmap_ftl\CMAP_FTL"                # pacote functions (só leitura)
PASTA_SAIDA      = rf"{PASTA_PROJETO}\resultados\4_cmap\{NOME_INFERENCIA}"

ANOS = [2005, 2006, 2007, 2008, 2009, 2010]
ANOS_REFERENCIA = [2008, 2010]    # anos com TerraClass
CLASSIFICADORES = None            # None = todos os do 3_inferencia, na ordem de versao_classificadores.csv

REFAZER = False                   # False: reaproveita CMAP e data a data já gravados (permite retomar)
TAMANHO_BLOCO_PX = 50             # blocos do bootstrap espacial (50 px = 1,5 km)
N_BOOTSTRAP = 1000
SEMENTE = 42
AREA_PIXEL_HA = 0.09              # pixel de 30 m
N_FAIXAS_CONFIANCA = 10           # faixas de probabilidade na avaliação da confiança (seção 6)

## 1. Legenda, matrizes, máscaras e referência

Os códigos de saída do CMAP são os da legenda do 0b (F = 1, VS = 2, PA = 3, AG = 4), os mesmos do TerraClass recodificado no 0c.

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

sys.path.insert(0, PASTA_REPO)
from functions.cmap import cmap_classifier, maxver_from_likelihoods
from functions.transitions import read_transition_matrix, trajectory_priors
from functions.trajectory_assessment import impossible_trajectories, compare_classifications

if not os.path.abspath(PASTA_SAIDA).lower().startswith(os.path.abspath(PASTA_PROJETO).lower() + os.sep):
    raise ValueError(f"PASTA_SAIDA precisa ficar dentro de {PASTA_PROJETO}")
os.makedirs(PASTA_SAIDA, exist_ok=True)
with open(os.path.join(PASTA_CONJUNTO, "legenda.json"), encoding="utf-8") as arq:
    legenda = json.load(arq)
CLASSES = {int(k): v for k, v in legenda["classes"].items()}     # código -> sigla
CODIGOS = sorted(CLASSES)
SIGLAS = [CLASSES[c] for c in CODIGOS]
CODIGOS_CMAP = {CLASSES[c]: c for c in CODIGOS}                  # sigla -> código (saída do CMAP)
K = len(CODIGOS)

# ---- matrizes de transição e tabela de trajetórias
arquivos_matrizes = [os.path.join(PASTA_MATRIZES, f"transicao_{a1}_{a2}.csv") for a1, a2 in zip(ANOS[:-1], ANOS[1:])]
matrizes = [read_transition_matrix(a, verbose=False) for a in arquivos_matrizes]
tabela_trajetorias = trajectory_priors(matrizes)
tabela_trajetorias.to_csv(os.path.join(PASTA_SAIDA, "tabela_trajetorias.csv"), sep=";", index_label="Trajectory_ID")
n_validas = int((tabela_trajetorias["final_weight"] > 0).sum())
print(f"Trajetórias: {len(tabela_trajetorias)} | válidas: {n_validas}")

# ---- classificadores e máscaras do 3_inferencia (1 = sem probabilidade na data)
if CLASSIFICADORES is None:
    CLASSIFICADORES = pd.read_csv(os.path.join(PASTA_INFERENCIA, "versao_classificadores.csv"))["classificador"].tolist()
masks = [os.path.join(PASTA_INFERENCIA, f"mascara_inferencia_{ano}.tif") for ano in ANOS]
for nome in CLASSIFICADORES:
    faltam = [ano for ano in ANOS if not os.path.exists(os.path.join(PASTA_INFERENCIA, nome, f"logp_{ano}.tif"))]
    if faltam:
        raise FileNotFoundError(f"{nome}: faltam as probabilidades de {faltam} em {PASTA_INFERENCIA}")
print("Classificadores:", CLASSIFICADORES)

# ---- referência TerraClass (0c) e distância à borda
ref, dist_ref = {}, {}
for ano in ANOS_REFERENCIA:
    with rasterio.open(os.path.join(PASTA_REFERENCIA, f"referencia_{ano}.tif")) as src:
        ref[ano] = src.read(1)
        grade = {"crs": src.crs, "transform": src.transform, "height": src.height, "width": src.width}
    with rasterio.open(os.path.join(PASTA_REFERENCIA, f"dist_borda_{ano}.tif")) as src:
        dist_ref[ano] = src.read(1)
with rasterio.open(masks[0]) as src:
    if (src.transform, src.height, src.width) != (grade["transform"], grade["height"], grade["width"]):
        raise ValueError("A referência do 0c e as saídas do 3_inferencia não estão na mesma grade")
H, W = grade["height"], grade["width"]
for ano in ANOS_REFERENCIA:
    print(f"TerraClass {ano}: {int(np.isin(ref[ano], CODIGOS).sum())} pixels na legenda "
          f"({100 * np.isin(ref[ano], CODIGOS).mean():.1f}% da grade)")

## 2. CMAP e classificação data a data

Para cada classificador, grava em `<classificador>\`: `CMAP.tif` (uma banda por ano), `Trajectory_IDs.tif` (trajetória escolhida, índice de `tabela_trajetorias.csv`), `Log_Posterior.tif` e `data_a_data.tif`.

In [ ]:
inicio = time.time()
tabelas_traj = {}
for nome in CLASSIFICADORES:
    pasta = os.path.join(PASTA_SAIDA, nome)
    arq_cmap, arq_dd = os.path.join(pasta, "CMAP.tif"), os.path.join(pasta, "data_a_data.tif")
    tifs = [os.path.join(PASTA_INFERENCIA, nome, f"logp_{ano}.tif") for ano in ANOS]
    if REFAZER or not (os.path.exists(arq_cmap) and os.path.exists(arq_dd)):
        t0 = time.time()
        saida_cmap = cmap_classifier(tifs, tabela_trajetorias, output_dir=pasta, date_labels=ANOS, class_codes=CODIGOS_CMAP,
                                     input_is_log=True, masks=masks, mask_mode="ignore", output_layout="stack",
                                     save_trajectory_ids=True, save_log_posterior=True, verbose=False)
        maxver_from_likelihoods(tifs, arq_dd, CODIGOS_CMAP, date_labels=ANOS, input_is_log=True, masks=masks)
        saida_cmap["trajectory_table"].to_csv(os.path.join(pasta, "trajetorias_escolhidas.csv"), sep=";", index=False)
        print(f"{nome}: classificado em {time.time() - t0:.0f} s")
    else:
        print(f"{nome}: já existe")
    tabelas_traj[nome] = pd.read_csv(os.path.join(pasta, "trajetorias_escolhidas.csv"), sep=";")
print(f"Total: {(time.time() - inicio) / 60:.1f} min")

## 3. Trajetórias inválidas, discordância e áreas

- **Trajetórias inválidas no data a data**: proporção dos pixels classificados nas 6 datas cuja sequência de classes tem alguma transição proibida (volta para Floresta). No CMAP deve ser 0, o que confere o próprio CMAP.
- **Discordância entre data a data e CMAP**: por ano e na trajetória (pixels com pelo menos um ano diferente), como na Tabela 3 de Reis et al. (2020).
- **Áreas**: proporção de cada classe por ano no CMAP e no data a data.
- **Trajetórias mais frequentes no CMAP**.

In [ ]:
linhas_cena, linhas_area, mapas, invalidos_dd, mudancas, trajetorias_top = [], [], {}, {}, {}, []
for nome in CLASSIFICADORES:
    pasta = os.path.join(PASTA_SAIDA, nome)
    arq_cmap, arq_dd = os.path.join(pasta, "CMAP.tif"), os.path.join(pasta, "data_a_data.tif")
    inv_dd = impossible_trajectories(arq_dd, matrizes, legends=CLASSES, exclude_codes=[0],
                                     output_path=os.path.join(pasta, "trajetorias_invalidas_data_a_data.tif"))
    inv_cmap = impossible_trajectories(arq_cmap, matrizes, legends=CLASSES, exclude_codes=[0])
    comp = compare_classifications(arq_dd, arq_cmap, legend_a=CLASSES, legend_b=CLASSES, date_labels=[str(a) for a in ANOS],
                                   output_path=os.path.join(pasta, "anos_alterados_pelo_cmap.tif"), verbose=False)
    linha = {"classificador": nome, "pct_trajetorias_invalidas_data_a_data": inv_dd["percentage"],
             "pct_trajetorias_invalidas_cmap": inv_cmap["percentage"], "n_pixels_6_datas": inv_dd["n_valid"]}
    for _, r_ in comp["per_date"].iterrows():
        linha[f"discordancia_{r_['date']}"] = r_["percentage"]
    linha["discordancia_trajetoria"] = comp["total_percentage"]
    linhas_cena.append(linha)
    with rasterio.open(os.path.join(pasta, "trajetorias_invalidas_data_a_data.tif")) as src:
        invalidos_dd[nome] = src.read(1)
    mudancas[nome] = comp["accumulated"]
    with rasterio.open(arq_cmap) as src:
        mapas[nome] = src.read()
    with rasterio.open(arq_dd) as src:
        dd = src.read()
    for metodo, arr in (("CMAP", mapas[nome]), ("data_a_data", dd)):
        for t, ano in enumerate(ANOS):
            v = arr[t]
            n_cl = int((v > 0).sum())
            for c in CODIGOS:
                linhas_area.append({"classificador": nome, "metodo": metodo, "ano": ano, "classe": CLASSES[c],
                                    "pct": 100 * (v == c).sum() / n_cl, "area_ha": (v == c).sum() * AREA_PIXEL_HA})
    t = tabelas_traj[nome]
    t = t[t["n_pixels"] > 0].sort_values("n_pixels", ascending=False)
    stage_cols = [c for c in t.columns if c.startswith("Stage_")]
    for _, r_ in t.head(15).iterrows():
        trajetorias_top.append({"classificador": nome, "trajetoria": " ".join(r_[stage_cols]), "n_pixels": int(r_["n_pixels"]),
                                "pct": 100 * r_["n_pixels"] / t["n_pixels"].sum()})

tab_cena = pd.DataFrame(linhas_cena).set_index("classificador")
tab_area = pd.DataFrame(linhas_area)
tab_top = pd.DataFrame(trajetorias_top)
tab_cena.to_csv(os.path.join(PASTA_SAIDA, "resumo_cena.csv"))
tab_area.to_csv(os.path.join(PASTA_SAIDA, "areas_classes.csv"), index=False)
tab_top.to_csv(os.path.join(PASTA_SAIDA, "trajetorias_mais_frequentes.csv"), index=False)
print("Trajetórias inválidas (%) e discordância data a data × CMAP (%):")
display(tab_cena.round(2))
print("Proporção de cada classe (%) por ano:")
display(tab_area.pivot_table(index=["classificador", "metodo", "ano"], columns="classe", values="pct")[SIGLAS].round(1))
print("Trajetórias mais frequentes no CMAP (5 primeiras de cada classificador):")
display(tab_top.groupby("classificador", sort=False).head(5).round(2))

## 4. Avaliação contra o TerraClass 2008 e 2010

Para cada classificador, método (data a data ou CMAP), ano com referência e recorte (`todos`: todos os pixels da referência dentro da legenda; `sem_borda`: distância à borda da referência ≥ 2): acurácia global, kappa, acurácia balanceada e acurácias do produtor (AP) e do usuário (AU) por classe. Linhas da matriz de confusão = TerraClass, colunas = classificação. Pixel sem classe na data (máscara) fica fora da conta.

**Par 2008–2010** (pixels com referência e classe nos dois anos):
- `acerto_par`: fração em que as duas datas estão corretas;
- `mudancas_detectadas`: entre os pixels que mudaram de classe no TerraClass, fração em que a classificação também mudou;
- `mudancas_falsas`: entre os pixels que não mudaram no TerraClass, fração em que a classificação mudou;
- `precisao_mudanca`: entre os pixels que mudaram na classificação, fração que também mudou no TerraClass;
- `acerto_transicao`: entre os pixels que mudaram no TerraClass, fração com as duas classes corretas.

In [ ]:
n_bl_lin, n_bl_col = -(-H // TAMANHO_BLOCO_PX), -(-W // TAMANHO_BLOCO_PX)
n_blocos = n_bl_lin * n_bl_col
linhas_idx, colunas_idx = np.indices((H, W))
bloco = (linhas_idx // TAMANHO_BLOCO_PX) * n_bl_col + (colunas_idx // TAMANHO_BLOCO_PX)
cm_blocos, linhas_av, linhas_par = {}, [], []
for nome in CLASSIFICADORES:
    with rasterio.open(os.path.join(PASTA_SAIDA, nome, "data_a_data.tif")) as src:
        dd = src.read()
    for metodo, arr in (("data_a_data", dd), ("CMAP", mapas[nome])):
        for ano in ANOS_REFERENCIA:
            pred_ano = arr[ANOS.index(ano)]
            for recorte in ("todos", "sem_borda"):
                sel = np.isin(ref[ano], CODIGOS) & (pred_ano > 0)
                if recorte == "sem_borda":
                    sel &= dist_ref[ano] >= 2
                y_ref = np.searchsorted(CODIGOS, ref[ano][sel])
                y_pred = np.searchsorted(CODIGOS, pred_ano[sel])
                cm = np.zeros((K, K), dtype=np.int64)
                np.add.at(cm, (y_ref, y_pred), 1)
                if recorte == "todos":
                    cb = np.zeros((n_blocos, K, K), dtype=np.int64)
                    np.add.at(cb, (bloco[sel], y_ref, y_pred), 1)
                    cm_blocos[(nome, metodo, ano)] = cb
                    pd.DataFrame(cm, index=[f"ref_{s}" for s in SIGLAS], columns=[f"cls_{s}" for s in SIGLAS]).to_csv(
                        os.path.join(PASTA_SAIDA, nome, f"confusao_{metodo}_{ano}.csv"))
                n_tot = cm.sum()
                oa = np.trace(cm) / n_tot
                pe = (cm.sum(axis=0) * cm.sum(axis=1)).sum() / n_tot ** 2
                n_r, n_p = cm.sum(axis=1), cm.sum(axis=0)
                ap = np.where(n_r > 0, np.diag(cm) / np.maximum(n_r, 1), np.nan)
                au = np.where(n_p > 0, np.diag(cm) / np.maximum(n_p, 1), np.nan)
                linha = {"classificador": nome, "metodo": metodo, "ano": ano, "recorte": recorte, "n": int(n_tot),
                         "acuracia_global": oa, "kappa": (oa - pe) / (1 - pe), "acuracia_balanceada": np.nanmean(ap[n_r > 0])}
                for k in range(K):
                    linha[f"AP_{SIGLAS[k]}"], linha[f"AU_{SIGLAS[k]}"] = ap[k], au[k]
                linhas_av.append(linha)

        # par 2008-2010
        a0, a1 = ANOS_REFERENCIA[0], ANOS_REFERENCIA[-1]
        p0, p1 = arr[ANOS.index(a0)], arr[ANOS.index(a1)]
        sel = np.isin(ref[a0], CODIGOS) & np.isin(ref[a1], CODIGOS) & (p0 > 0) & (p1 > 0)
        r0, r1, q0, q1 = ref[a0][sel], ref[a1][sel], p0[sel], p1[sel]
        ref_mudou, cls_mudou = r0 != r1, q0 != q1
        linhas_par.append({"classificador": nome, "metodo": metodo, "n": int(sel.sum()),
                           "acerto_par": np.mean((q0 == r0) & (q1 == r1)),
                           "pct_mudou_terraclass": 100 * ref_mudou.mean(), "pct_mudou_classificacao": 100 * cls_mudou.mean(),
                           "mudancas_detectadas": cls_mudou[ref_mudou].mean(), "mudancas_falsas": cls_mudou[~ref_mudou].mean(),
                           "precisao_mudanca": ref_mudou[cls_mudou].mean() if cls_mudou.any() else np.nan,
                           "acerto_transicao": np.mean((q0 == r0)[ref_mudou] & (q1 == r1)[ref_mudou])})

tab_av = pd.DataFrame(linhas_av)
tab_par = pd.DataFrame(linhas_par)
tab_av.to_csv(os.path.join(PASTA_SAIDA, "avaliacao_terraclass.csv"), index=False)
tab_par.to_csv(os.path.join(PASTA_SAIDA, "avaliacao_par_2008_2010.csv"), index=False)
cols = ["acuracia_global", "kappa", "acuracia_balanceada"] + [f"AP_{s}" for s in SIGLAS] + [f"AU_{s}" for s in SIGLAS]
ordem = {n: i for i, n in enumerate(CLASSIFICADORES)}
for recorte in ("todos", "sem_borda"):
    print(f"\nRecorte: {recorte}")
    t = tab_av[tab_av.recorte == recorte].sort_values(["ano", "classificador", "metodo"], key=lambda s: s.map(ordem) if s.name == "classificador" else s)
    display(t.set_index(["ano", "classificador", "metodo"])[cols].round(3))
print("\nPar 2008-2010:")
display(tab_par.set_index(["classificador", "metodo"]).round(3))

## 5. Intervalos e comparações pareadas (bootstrap por blocos)

Em cada réplica, sorteiam-se com reposição `n_blocos` blocos de `TAMANHO_BLOCO_PX` × `TAMANHO_BLOCO_PX` pixels; as matrizes de confusão dos blocos sorteados são somadas e as métricas, recalculadas. Os **mesmos** blocos valem para todas as classificações da réplica, o que torna as diferenças pareadas.

- Primeira tabela: intervalo de 95% (percentis 2,5 e 97,5) de cada métrica, recorte `todos`.
- Segunda tabela: diferenças **CMAP − data a data** em cada classificador e **CNN − outro classificador** (os dois com CMAP), com intervalo de 95% e a fração de réplicas em que a diferença é positiva.

In [ ]:
rng = np.random.default_rng(SEMENTE)
chaves = list(cm_blocos)
boot = {ch: np.empty((N_BOOTSTRAP, 3)) for ch in chaves}      # acurácia global, kappa, acurácia balanceada
for b in range(N_BOOTSTRAP):
    pesos = np.bincount(rng.integers(0, n_blocos, n_blocos), minlength=n_blocos)
    for ch in chaves:
        cm = np.tensordot(pesos, cm_blocos[ch], axes=1).astype(float)
        n_tot = cm.sum()
        oa = np.trace(cm) / n_tot
        pe = (cm.sum(axis=0) * cm.sum(axis=1)).sum() / n_tot ** 2
        n_r = cm.sum(axis=1)
        boot[ch][b] = [oa, (oa - pe) / (1 - pe), np.mean(np.diag(cm)[n_r > 0] / n_r[n_r > 0])]
METRICAS_BOOT = ["acuracia_global", "kappa", "acuracia_balanceada"]

linhas = []
for (nome, metodo, ano), v in boot.items():
    for i, mt in enumerate(METRICAS_BOOT):
        linhas.append({"classificador": nome, "metodo": metodo, "ano": ano, "metrica": mt,
                       "ic_inf": np.percentile(v[:, i], 2.5), "ic_sup": np.percentile(v[:, i], 97.5)})
tab_ic = pd.DataFrame(linhas)
tab_ic.to_csv(os.path.join(PASTA_SAIDA, "intervalos_bootstrap.csv"), index=False)

linhas = []
for ano in ANOS_REFERENCIA:
    for nome in CLASSIFICADORES:
        d = boot[(nome, "CMAP", ano)] - boot[(nome, "data_a_data", ano)]
        pontual = tab_av[(tab_av.classificador == nome) & (tab_av.ano == ano) & (tab_av.recorte == "todos")].set_index("metodo")
        for i, mt in enumerate(METRICAS_BOOT):
            linhas.append({"comparacao": f"{nome}: CMAP − data a data", "ano": ano, "metrica": mt,
                           "diferenca": pontual.loc["CMAP", mt] - pontual.loc["data_a_data", mt],
                           "ic_inf": np.percentile(d[:, i], 2.5), "ic_sup": np.percentile(d[:, i], 97.5),
                           "frac_replicas_positivas": (d[:, i] > 0).mean()})
    if "CNN" in CLASSIFICADORES:
        for nome in CLASSIFICADORES:
            if nome == "CNN":
                continue
            d = boot[("CNN", "CMAP", ano)] - boot[(nome, "CMAP", ano)]
            pa = tab_av[(tab_av.ano == ano) & (tab_av.recorte == "todos") & (tab_av.metodo == "CMAP")].set_index("classificador")
            for i, mt in enumerate(METRICAS_BOOT):
                linhas.append({"comparacao": f"CMAP: CNN − {nome}", "ano": ano, "metrica": mt,
                               "diferenca": pa.loc["CNN", mt] - pa.loc[nome, mt],
                               "ic_inf": np.percentile(d[:, i], 2.5), "ic_sup": np.percentile(d[:, i], 97.5),
                               "frac_replicas_positivas": (d[:, i] > 0).mean()})
tab_dif = pd.DataFrame(linhas)
tab_dif.to_csv(os.path.join(PASTA_SAIDA, "comparacoes_pareadas.csv"), index=False)
print(f"{n_blocos} blocos de {TAMANHO_BLOCO_PX} x {TAMANHO_BLOCO_PX} px | {N_BOOTSTRAP} réplicas")
print("\nIntervalos de 95%:")
display(tab_ic.pivot_table(index=["ano", "classificador", "metodo"], columns="metrica", values=["ic_inf", "ic_sup"]).round(3))
print("\nComparações pareadas:")
display(tab_dif.set_index(["comparacao", "ano", "metrica"]).round(3))

## 6. Confiança das trajetórias

O log-posterior de uma trajetória s é a soma das log-probabilidades de cada data mais o log de P(s), o mesmo que o CMAP maximiza. A **confiança da trajetória** é a probabilidade a posteriori da trajetória escolhida, normalizada por todas as trajetórias válidas:

P(s* | X) = exp(log-posterior de s*) / Σ exp(log-posterior de s).

Como P(s) é um produto de matrizes de transição (cadeia de Markov), a soma sobre as 1.093 trajetórias válidas sai do **algoritmo forward**, sem enumerá-las. Com o **forward-backward** sai também a **probabilidade marginal da classe do CMAP em cada data**, P(classe do CMAP na data t | X), somando todas as trajetórias que passam por ela.

- O modelo é exatamente o do CMAP: as mesmas probabilidades por data, as datas mascaradas sem informação (`mask_mode="ignore"`) e as mesmas matrizes. Conferência: o máximo (Viterbi) deve ser igual ao `Log_Posterior.tif` gravado pelo CMAP.
- A confiança da trajetória é calculada nos pixels com as 6 datas observadas; a marginal, em cada data com classe.
- As probabilidades de entrada foram calibradas nos folds do Pará, com classes equiprováveis; a área de estudo tem outra composição (73% de Floresta). Por isso a confiança é a do modelo, e a calibração na área de estudo é **conferida contra o TerraClass** 2008 e 2010:
  - **por data**: acerto por faixa de probabilidade (diagrama de confiabilidade) e erro de calibração (ECE), para a marginal do CMAP e para a probabilidade máxima do data a data;
  - **trajetória**: acerto das duas datas (2008 e 2010) por faixa de confiança; acerto nos pixels muito confiáveis (≥ 0,9) e pouco confiáveis (< 0,5); fração dos erros que está entre os 20% de pixels menos confiáveis;
  - **trajetórias inválidas**: confiança média onde o data a data era inválido e onde era válido.

In [ ]:
from scipy.special import logsumexp

FAIXAS = np.linspace(0, 1, N_FAIXAS_CONFIANCA + 1)
log_M = []                                                       # matrizes do CMAP em log (0 -> -inf), ordem da legenda
for m in matrizes:
    with np.errstate(divide="ignore"):
        log_M.append(np.log(m.loc[SIGLAS, SIGLAS].to_numpy(dtype=float)))
mascaras = np.empty((len(ANOS), grade["height"], grade["width"]), dtype=bool)
for t, arq in enumerate(masks):
    with rasterio.open(arq) as src:
        mascaras[t] = src.read(1) == 1
seis_datas = ~mascaras.any(axis=0)
perfil_conf = {"driver": "GTiff", "dtype": "float32", "nodata": np.nan, "crs": grade["crs"], "transform": grade["transform"],
               "height": grade["height"], "width": grade["width"], "compress": "deflate"}
t8, t10 = ANOS.index(ANOS_REFERENCIA[0]), ANOS.index(ANOS_REFERENCIA[-1])
sel_par = seis_datas & np.isin(ref[ANOS_REFERENCIA[0]], CODIGOS) & np.isin(ref[ANOS_REFERENCIA[-1]], CODIGOS)

linhas_resumo, linhas_cal, linhas_traj_faixa, conf_traj = [], [], [], {}
for nome in CLASSIFICADORES:
    pasta = os.path.join(PASTA_SAIDA, nome)
    # ---- log-probabilidades por data, na ordem da legenda; data mascarada = sem informação (0)
    L = np.empty((len(ANOS), K, grade["height"], grade["width"]))
    for t, ano in enumerate(ANOS):
        with rasterio.open(os.path.join(PASTA_INFERENCIA, nome, f"logp_{ano}.tif")) as src:
            descricoes = list(src.descriptions)
            d = src.read().astype(np.float64)
        L[t] = d[[descricoes.index(s) for s in SIGLAS]] if all(descricoes) else d
        L[t][:, mascaras[t]] = 0.0
    L[~np.isfinite(L)] = 0.0                                     # fora da imagem (sem classe no CMAP)

    # ---- forward (soma) e Viterbi (máximo)
    alfa = np.empty_like(L)
    alfa[0] = L[0]
    vit = L[0].copy()
    for t in range(1, len(ANOS)):
        alfa[t] = L[t] + logsumexp(alfa[t - 1][:, None] + log_M[t - 1][:, :, None, None], axis=0)
        vit = L[t] + np.max(vit[:, None] + log_M[t - 1][:, :, None, None], axis=0)
    log_Z = logsumexp(alfa[-1], axis=0)
    log_max = vit.max(axis=0)
    # ---- backward e marginais
    beta = np.zeros_like(L)
    for t in range(len(ANOS) - 2, -1, -1):
        beta[t] = logsumexp(log_M[t][:, :, None, None] + (L[t + 1] + beta[t + 1])[None], axis=1)
    marginal = np.exp(alfa + beta - log_Z)                       # (datas, classes, linhas, colunas)
    del alfa, beta

    # ---- conferência com o CMAP e mapas de confiança
    with rasterio.open(os.path.join(pasta, "Log_Posterior.tif")) as src:
        lp_cmap = src.read(1)
    ok = np.isfinite(lp_cmap)
    dif_viterbi = float(np.abs(log_max[ok] - lp_cmap[ok]).max())
    conf = np.where(seis_datas & ok, np.exp(log_max - log_Z), np.nan)
    conf_traj[nome] = conf
    classe_cmap = mapas[nome]
    conf_data = np.full(classe_cmap.shape, np.nan)
    for t in range(len(ANOS)):
        tem = classe_cmap[t] > 0
        k = np.searchsorted(CODIGOS, classe_cmap[t][tem])
        conf_data[t][tem] = marginal[t][k, np.nonzero(tem)[0], np.nonzero(tem)[1]]
    with rasterio.open(os.path.join(pasta, "confianca_trajetoria.tif"), "w", count=1, **perfil_conf) as dst:
        dst.write(conf.astype(np.float32), 1)
        dst.set_band_description(1, "P(trajetoria do CMAP | X)")
    with rasterio.open(os.path.join(pasta, "confianca_por_data.tif"), "w", count=len(ANOS), **perfil_conf) as dst:
        dst.write(conf_data.astype(np.float32))
        for t, ano in enumerate(ANOS):
            dst.set_band_description(t + 1, str(ano))

    # ---- calibração por data (2008 e 2010): marginal do CMAP x probabilidade máxima do data a data
    with rasterio.open(os.path.join(pasta, "data_a_data.tif")) as src:
        dd = src.read()
    ece = {}
    for metodo in ("CMAP (marginal)", "data a data (máxima)"):
        p_, y_ = [], []
        for ano in ANOS_REFERENCIA:
            t = ANOS.index(ano)
            if metodo == "CMAP (marginal)":
                sel = (classe_cmap[t] > 0) & np.isin(ref[ano], CODIGOS)
                p_.append(conf_data[t][sel]); y_.append((classe_cmap[t] == ref[ano])[sel])
            else:
                sel = (dd[t] > 0) & np.isin(ref[ano], CODIGOS)
                p_.append(np.exp(L[t].max(axis=0))[sel]); y_.append((dd[t] == ref[ano])[sel])
        p_, y_ = np.concatenate(p_), np.concatenate(y_)
        faixa = np.clip(np.digitize(p_, FAIXAS) - 1, 0, N_FAIXAS_CONFIANCA - 1)
        ece[metodo] = 0.0
        for b in range(N_FAIXAS_CONFIANCA):
            nb = faixa == b
            if nb.any():
                ece[metodo] += nb.mean() * abs(y_[nb].mean() - p_[nb].mean())
                linhas_cal.append({"classificador": nome, "metodo": metodo, "faixa_inf": FAIXAS[b], "faixa_sup": FAIXAS[b + 1],
                                   "pct_pixels": 100 * nb.mean(), "confianca_media": p_[nb].mean(), "acerto": y_[nb].mean()})

    # ---- confiança da trajetória x acerto das duas datas
    par_certo = ((classe_cmap[t8] == ref[ANOS_REFERENCIA[0]]) & (classe_cmap[t10] == ref[ANOS_REFERENCIA[-1]]))[sel_par]
    cf = conf[sel_par]
    faixa = np.clip(np.digitize(cf, FAIXAS) - 1, 0, N_FAIXAS_CONFIANCA - 1)
    for b in range(N_FAIXAS_CONFIANCA):
        nb = faixa == b
        if nb.any():
            linhas_traj_faixa.append({"classificador": nome, "faixa_inf": FAIXAS[b], "faixa_sup": FAIXAS[b + 1],
                                      "pct_pixels": 100 * nb.mean(), "confianca_media": cf[nb].mean(), "acerto_par": par_certo[nb].mean()})
    q20 = np.quantile(cf, 0.2)
    alta, baixa = cf >= 0.9, cf < 0.5
    inval = (invalidos_dd[nome] == 1) & seis_datas
    valid = (invalidos_dd[nome] == 0) & seis_datas
    linhas_resumo.append({"classificador": nome, "confianca_trajetoria_media": np.nanmean(conf), "confianca_trajetoria_mediana": np.nanmedian(conf),
                          "pct_pixels_confianca_ge_0.9": 100 * alta.mean(), "acerto_par_confianca_ge_0.9": par_certo[alta].mean() if alta.any() else np.nan,
                          "pct_pixels_confianca_lt_0.5": 100 * baixa.mean(), "acerto_par_confianca_lt_0.5": par_certo[baixa].mean() if baixa.any() else np.nan,
                          "acerto_par_todos": par_certo.mean(), "pct_erros_nos_20pct_menos_confiaveis": 100 * (~par_certo & (cf <= q20)).sum() / max((~par_certo).sum(), 1),
                          "confianca_media_dd_invalida": np.nanmean(conf[inval]), "confianca_media_dd_valida": np.nanmean(conf[valid]),
                          "ece_cmap_marginal": ece["CMAP (marginal)"], "ece_data_a_data": ece["data a data (máxima)"],
                          "dif_max_viterbi_log_posterior": dif_viterbi})
    del L, marginal
    print(f"{nome}: confiança calculada (Viterbi x Log_Posterior.tif: diferença máxima {dif_viterbi:.1e})")

tab_conf = pd.DataFrame(linhas_resumo).set_index("classificador")
tab_cal = pd.DataFrame(linhas_cal)
tab_traj_faixa = pd.DataFrame(linhas_traj_faixa)
tab_conf.to_csv(os.path.join(PASTA_SAIDA, "confianca_resumo.csv"))
tab_cal.to_csv(os.path.join(PASTA_SAIDA, "confianca_calibracao_por_data.csv"), index=False)
tab_traj_faixa.to_csv(os.path.join(PASTA_SAIDA, "confianca_trajetoria_por_faixa.csv"), index=False)
print("\nResumo (acerto_par = 2008 e 2010 corretos):")
display(tab_conf.T.round(3))
print("\nAcerto das duas datas por faixa de confiança da trajetória:")
display(tab_traj_faixa.pivot(index=["faixa_inf", "faixa_sup"], columns="classificador", values=["pct_pixels", "acerto_par"])
        .reindex(columns=CLASSIFICADORES, level=1).round(3))

# ---- figuras: mapas de confiança e calibração
fig, eixos = plt.subplots(1, len(CLASSIFICADORES), figsize=(3.5 * len(CLASSIFICADORES), 4.8), squeeze=False)
cm_conf = plt.get_cmap("Blues").copy()
cm_conf.set_bad("#f0f0f0")
for j, nome in enumerate(CLASSIFICADORES):
    ax = eixos[0, j]
    im = ax.imshow(conf_traj[nome], cmap=cm_conf, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"{nome}\nmediana {tab_conf.loc[nome, 'confianca_trajetoria_mediana']:.2f} · ≥ 0,9: "
                 f"{tab_conf.loc[nome, 'pct_pixels_confianca_ge_0.9']:.0f}% dos pixels", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=eixos[0, :].tolist(), shrink=0.7, label="P(trajetória do CMAP | X)")
fig.savefig(os.path.join(PASTA_SAIDA, "confianca_trajetorias.png"), dpi=150, bbox_inches="tight")
plt.show()

COR_CLASSIFICADOR = dict(zip(CLASSIFICADORES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
fig, eixos = plt.subplots(1, 2, figsize=(11, 4.6))
for nome in CLASSIFICADORES:
    t = tab_cal[(tab_cal.classificador == nome) & (tab_cal.metodo == "CMAP (marginal)") & (tab_cal.pct_pixels >= 0.5)]
    eixos[0].plot(t["confianca_media"], t["acerto"], marker="o", markersize=4, linewidth=2, color=COR_CLASSIFICADOR[nome],
                  label=f"{nome} (ECE {tab_conf.loc[nome, 'ece_cmap_marginal']:.3f})")
    t = tab_traj_faixa[(tab_traj_faixa.classificador == nome) & (tab_traj_faixa.pct_pixels >= 0.5)]
    eixos[1].plot(t["confianca_media"], t["acerto_par"], marker="o", markersize=4, linewidth=2, color=COR_CLASSIFICADOR[nome], label=nome)
eixos[0].plot([0, 1], [0, 1], color="#8a8984", linestyle="--", linewidth=1)
eixos[0].set_xlabel("Probabilidade da classe do CMAP na data")
eixos[0].set_ylabel(f"Acerto contra o TerraClass ({ANOS_REFERENCIA[0]} e {ANOS_REFERENCIA[-1]})")
eixos[0].set_title("Calibração por data (faixas com ≥ 0,5% dos pixels)", fontsize=10, loc="left")
eixos[0].legend(fontsize=8, frameon=False)
eixos[1].set_xlabel("Confiança da trajetória, P(s* | X)")
eixos[1].set_ylabel(f"Acerto de {ANOS_REFERENCIA[0]} e {ANOS_REFERENCIA[-1]} juntos")
eixos[1].set_title("Confiança da trajetória x acerto", fontsize=10, loc="left")
for ax in eixos:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.grid(color="#e4e3de", linewidth=0.8); ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(PASTA_SAIDA, "confianca_calibracao.png"), dpi=150)
plt.show()

## 7. Figuras

In [ ]:
CORES_CLASSES = {"F": "#1b7837", "VS": "#a6dba0", "PA": "#e6ab02", "AG": "#d95f02"}
cores = ListedColormap(["white"] + [CORES_CLASSES.get(s, "#999999") for s in SIGLAS])
linhas_fig = list(CLASSIFICADORES) + ["TerraClass"]
fig, eixos = plt.subplots(len(linhas_fig), len(ANOS), figsize=(2.0 * len(ANOS), 3.1 * len(linhas_fig)), squeeze=False)
for i, nome in enumerate(linhas_fig):
    for j, ano in enumerate(ANOS):
        ax = eixos[i, j]
        ax.set_xticks([]); ax.set_yticks([])
        if nome == "TerraClass":
            if ano in ref:
                ax.imshow(np.where(np.isin(ref[ano], CODIGOS), ref[ano], 0), cmap=cores, vmin=-0.5, vmax=K + 0.5, interpolation="nearest")
            else:
                ax.axis("off")
        else:
            ax.imshow(mapas[nome][j], cmap=cores, vmin=-0.5, vmax=K + 0.5, interpolation="nearest")
        if i == 0:
            ax.set_title(str(ano), fontsize=10)
        if nome != "TerraClass" and j == 0:
            ax.set_ylabel(f"{nome} + CMAP", fontsize=9)
        if nome == "TerraClass" and ano == ANOS_REFERENCIA[0]:
            ax.set_ylabel("TerraClass", fontsize=9)
fig.legend(handles=[Patch(color=CORES_CLASSES.get(s, "#999999"), label=s) for s in SIGLAS] + [Patch(facecolor="white", edgecolor="#8a8984", label="sem classe / fora da legenda")],
           loc="lower center", ncol=K + 1, frameon=False)
fig.tight_layout(rect=(0, 0.025, 1, 1))
fig.savefig(os.path.join(PASTA_SAIDA, "mapas_cmap.png"), dpi=130)
plt.show()

fig, eixos = plt.subplots(2, len(CLASSIFICADORES), figsize=(3.0 * len(CLASSIFICADORES), 7.4), squeeze=False)
for j, nome in enumerate(CLASSIFICADORES):
    inv = np.where(invalidos_dd[nome] == 255, np.nan, invalidos_dd[nome].astype(float))
    eixos[0, j].imshow(inv, cmap=ListedColormap(["#f0f0f0", "#cb181d"]), vmin=0, vmax=1, interpolation="nearest")
    eixos[0, j].set_title(f"{nome}\ninválidas no data a data: {tab_cena.loc[nome, 'pct_trajetorias_invalidas_data_a_data']:.1f}%", fontsize=8)
    mud = np.where(mudancas[nome] < 0, np.nan, mudancas[nome].astype(float))
    im = eixos[1, j].imshow(mud, cmap="magma_r", vmin=0, vmax=len(ANOS), interpolation="nearest")
    eixos[1, j].set_title(f"anos alterados pelo CMAP\n(≥ 1 ano: {tab_cena.loc[nome, 'discordancia_trajetoria']:.1f}% dos pixels)", fontsize=8)
    for ax in eixos[:, j]:
        ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=eixos[1, :].tolist(), shrink=0.6, label="número de anos")
fig.savefig(os.path.join(PASTA_SAIDA, "invalidas_e_alteracoes.png"), dpi=130, bbox_inches="tight")
plt.show()

# acurácias com intervalo de 95%: data a data x CMAP
COR_METODO = {"data_a_data": "#2a78d6", "CMAP": "#eb6834"}
fig, eixos = plt.subplots(2, len(ANOS_REFERENCIA), figsize=(5.2 * len(ANOS_REFERENCIA), 6.4), squeeze=False, sharey="row")
x = np.arange(len(CLASSIFICADORES))
for i, mt in enumerate(["acuracia_global", "acuracia_balanceada"]):
    for j, ano in enumerate(ANOS_REFERENCIA):
        ax = eixos[i, j]
        for d_, metodo in enumerate(["data_a_data", "CMAP"]):
            t = tab_av[(tab_av.ano == ano) & (tab_av.recorte == "todos") & (tab_av.metodo == metodo)].set_index("classificador").loc[CLASSIFICADORES]
            ic = tab_ic[(tab_ic.ano == ano) & (tab_ic.metodo == metodo) & (tab_ic.metrica == mt)].set_index("classificador").loc[CLASSIFICADORES]
            ax.bar(x + (d_ - 0.5) * 0.38, t[mt], width=0.34, color=COR_METODO[metodo], label="data a data" if metodo == "data_a_data" else "CMAP",
                   yerr=[t[mt] - ic["ic_inf"], ic["ic_sup"] - t[mt]], error_kw={"elinewidth": 1, "capsize": 2, "ecolor": "#52514e"})
        ax.set_xticks(x, CLASSIFICADORES, fontsize=8)
        ax.set_title(f"{ano}", fontsize=10, loc="left")
        ax.set_ylim(0.4, 1)
        ax.grid(axis="y", color="#e4e3de", linewidth=0.8)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
    eixos[i, 0].set_ylabel("Acurácia global" if mt == "acuracia_global" else "Acurácia balanceada")
alcas, rotulos = eixos[0, 0].get_legend_handles_labels()
fig.legend(alcas, rotulos, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(os.path.join(PASTA_SAIDA, "acuracias_cmap.png"), dpi=150)
plt.show()

## 8. Saídas

| Arquivo | Conteúdo |
|---|---|
| `<classificador>\CMAP.tif`, `Trajectory_IDs.tif`, `Log_Posterior.tif` | CMAP (uma banda por ano), trajetória escolhida e log-posterior |
| `<classificador>\data_a_data.tif` | classe mais provável de cada data, empilhada |
| `<classificador>\trajetorias_invalidas_data_a_data.tif`, `anos_alterados_pelo_cmap.tif` | 1 = trajetória inválida no data a data; número de anos em que o CMAP mudou a classe |
| `<classificador>\trajetorias_escolhidas.csv`, `confusao_<metodo>_<ano>.csv` | pixels por trajetória; matrizes de confusão contra o TerraClass |
| `tabela_trajetorias.csv` | trajetórias e P(s) (o `Trajectory_ID` é o índice desta tabela) |
| `resumo_cena.csv`, `areas_classes.csv`, `trajetorias_mais_frequentes.csv` | trajetórias inválidas, discordância, áreas e trajetórias |
| `avaliacao_terraclass.csv`, `avaliacao_par_2008_2010.csv` | avaliação por ano e do par 2008–2010 |
| `intervalos_bootstrap.csv`, `comparacoes_pareadas.csv` | intervalos de 95% e diferenças pareadas |
| `<classificador>\confianca_trajetoria.tif` | probabilidade a posteriori da trajetória escolhida pelo CMAP (pixels com as 6 datas) |
| `<classificador>\confianca_por_data.tif` | probabilidade marginal da classe do CMAP em cada data (uma banda por ano) |
| `confianca_resumo.csv`, `confianca_calibracao_por_data.csv`, `confianca_trajetoria_por_faixa.csv` | avaliação da confiança contra o TerraClass |
| `mapas_cmap.png`, `invalidas_e_alteracoes.png`, `acuracias_cmap.png`, `confianca_trajetorias.png`, `confianca_calibracao.png` | figuras |

Códigos dos mapas: F = 1, VS = 2, PA = 3, AG = 4; 0 = sem classe.